In [ ]:
"""
@
Auteurs:        Jeffrey Jason Boekstaaf, Tim Paulus van Croimvort en Haydar Eryörük
Studentnummers: 500460365, 500916516 en 500910901
Datum:          20-03-2026
Vak:            Beroepsproject 3.4
Opleiding:      Toegepaste Wiskunde & Data Science
School:         Hogeschool van Amsterdam
"""
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay, f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

data_test = pd.read_csv("data_test.csv")

# Haalt de Gradient Boosting classifier op uit de lokale map.
with open('pipelineSVM.pkl', 'rb') as file:
    gradient_boosting_pipe = pickle.load(file)

predicted = gradient_boosting_pipe.predict_proba(data_test)
y_train = LabelEncoder().fit_transform(data_test['Churn'])

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(y_train, predicted, cmap = 'Blues')
plt.show()


In [ ]:
# Plot 1
plt.figure(figsize = (12, 4))
plt.subplot(1, 3, 1)
sns.kdeplot(data_test, x = predicted[:], hue = 'Churn', fill = True)
plt.xlim(0, 1)
plt.title('Verdeling voorspelde kansen')

# Zet een classification report om in een rij van een dataframe
def make_row(cr):
    return {
        'Accuracy':  cr['accuracy'],
        'Recall':    cr['1']['recall'],
        'Precision': cr['1']['precision'],
        'F1 Score':  cr['1']['f1-score']
    }

thresholds = np.arange(0, 1, 0.01)
plot_data = pd.DataFrame([
    make_row(classification_report(
        y_train, 
        predicted[:] > t, 
        zero_division = True, 
        output_dict = True)) for t in thresholds])
plot_data.index = thresholds

# Plot 2
plt.subplot(1, 3, 2)
sns.lineplot(plot_data)
plt.plot([0, 1], [0, 0], 'k:', alpha = 0.5, label = 'Nul Acc.')
plt.xlabel('Threshold')
plt.legend()
plt.title('Metric Plot')

# Plot 3
ax = plt.subplot(1, 3, 3)
roc = RocCurveDisplay.from_predictions(
    y_train, 
    predicted[:], 
    name = '', 
    plot_chance_level = True,
    ax = ax)
plt.fill_between(roc.fpr, roc.tpr, color = '0.8')
plt.title('ROC curve')
plt.show()


In [ ]:
W = (((data_test['Seconds of Use'] / 60.0) * 0.2) + (data_test['Frequency of SMS'] * 0.07))
expected_profit = (1 - (0.25 * predicted)) * W
print(expected_profit.sum())


In [ ]:
# Hier worden de accuracy, f1, precision en recall scores berekend en gegeven.
acc = round(accuracy_score(y_train, predicted), 2)
rec = round(recall_score(y_train, predicted), 2)
pre = round(precision_score(y_train, predicted), 2)
fsco = round(f1_score(y_train, predicted), 2)
table = {"Name": ["Accuracy", "Recall", "Precision", "F1-score"],
        "Value": [acc, rec, pre, fsco]}
df = pd.DataFrame(table)
print(df)